In [1]:
from pathlib import Path

repo_path = Path("/content/loan-default-mlbd")

if not repo_path.exists():
    !git clone -b feat/keras-fastai https://github.com/kratipatel/loan-default-mlbd.git

%cd /content/loan-default-mlbd

!python -m src.prepare_data

Cloning into 'loan-default-mlbd'...
remote: Enumerating objects: 99, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 99 (delta 36), reused 64 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (99/99), 1.33 MiB | 5.88 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/loan-default-mlbd
reading: https://raw.githubusercontent.com/tiwvis/MLBD-Projects/main/data_sets/LoansData_sample.csv.gz
saved: /content/loan-default-mlbd/data/processed/loans_clean.parquet (86141, 71)
        size      mean
split                 
test   12922  0.187587
train  60298  0.187585
val    12921  0.187602


In [2]:
import time
import json
import numpy as np
import pandas as pd

from pathlib import Path
from fastai.tabular.all import *

from src.data import load_splits
from src.metrics import evaluate

import torch

print("fastai version:", __import__("fastai").__version__)
print("PyTorch version:", torch.__version__)
print("All libraries imported successfully!")

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


fastai version: 2.8.12
PyTorch version: 2.14.1+cpu
All libraries imported successfully!


In [3]:
path = "data/processed/loans_clean.parquet"

tr, va, te = load_splits(path)

print("Train shape:", tr.shape)
print("Validation shape:", va.shape)
print("Test shape:", te.shape)

print("\nTarget distribution:")
print(tr["default"].value_counts(normalize=True))

Train shape: (60298, 71)
Validation shape: (12921, 71)
Test shape: (12922, 71)

Target distribution:
default
0    0.812415
1    0.187585
Name: proportion, dtype: float64


In [4]:
# Target and columns
target = "default"

cat_names = [
    "grade", "sub_grade", "home_ownership",
    "verification_status", "purpose", "addr_state",
    "initial_list_status", "application_type",
    "disbursement_method"
]

cont_names = [
    col for col in tr.select_dtypes(include=np.number).columns
    if col not in [target]
]

# Combine train and validation for fixed split
train_df = tr.copy()
valid_df = va.copy()
test_df = te.copy()

train_df["is_valid"] = False
valid_df["is_valid"] = True

combined_df = pd.concat([train_df, valid_df], ignore_index=True)

splits = (
    list(range(len(train_df))),
    list(range(len(train_df), len(combined_df)))
)

# Fastai preprocessing
to = TabularPandas(
    combined_df,
    procs=[Categorify, FillMissing, Normalize],
    cat_names=cat_names,
    cont_names=cont_names,
    y_names=target,
    y_block=CategoryBlock(),
    splits=splits
)

print("Categorical features:", len(cat_names))
print("Continuous features:", len(cont_names))
print("Training rows:", len(to.train))
print("Validation rows:", len(to.valid))
print("Test rows:", len(test_df))

Categorical features: 19
Continuous features: 59
Training rows: 60298
Validation rows: 12921
Test rows: 12922


In [5]:
# Create DataLoaders
dls = to.dataloaders(bs=256)

# Build tabular neural network
learn = tabular_learner(
    dls,
    layers=[200, 100],
    metrics=accuracy
)

print("Fastai model created successfully!")
learn.summary()

Fastai model created successfully!


<div></div>

TabularModel (Input shape: 256 x 19)
Layer (type)         Output Shape         Param #    Trainable 
                     256 x 5             
Embedding                                 40         True      
____________________________________________________________________________
                     256 x 12            
Embedding                                 432        True      
____________________________________________________________________________
                     256 x 4             
Embedding                                 20         True      
____________________________________________________________________________
                     256 x 3             
Embedding                                 12         True      
____________________________________________________________________________
                     256 x 7             
Embedding                                 98         True      
_____________________________________________________________

In [6]:
# Train the model and measure training time
start_time = time.time()

learn.fit_one_cycle(
    10,
    1e-2
)

fastai_training_time = time.time() - start_time

print("Training time:", round(fastai_training_time, 2), "seconds")

epoch,train_loss,valid_loss,accuracy,time
0,0.453630,0.447343,0.811470,00:03
1,0.431506,0.431002,0.814411,00:02
2,0.423249,0.440000,0.813714,00:02
3,0.420383,0.429531,0.817352,00:02
4,0.413413,0.440465,0.817197,00:03
5,0.410912,0.430602,0.815958,00:02
6,0.402405,0.443223,0.812863,00:02
7,0.396059,0.448230,0.809380,00:02
8,0.397999,0.452292,0.811702,00:03
9,0.390377,0.458508,0.810773,00:02


Training time: 28.17 seconds


In [7]:
# Get validation probabilities
val_probs, val_targets = learn.get_preds(dl=dls.valid)

# Get test probabilities
test_dl = learn.dls.test_dl(test_df)
test_probs, _ = learn.get_preds(dl=test_dl)

# Extract probability of class 1 (default)
val_prob = val_probs[:, 1].numpy()
test_prob = test_probs[:, 1].numpy()

y_val = val_targets.numpy()
y_test = te[target].to_numpy()

# Calculate evaluation metrics
fastai_results = {
    "validation": evaluate(y_val, val_prob),
    "test": evaluate(y_test, test_prob),
    "training_time_seconds": round(fastai_training_time, 2)
}

print(json.dumps(fastai_results, indent=4))

{
    "validation": {
        "rmse": 0.3744712125422513,
        "mae": 0.26295547291870924,
        "r2": 0.0799078738487895,
        "roc_auc": 0.716903163594439,
        "pr_auc": 0.3655209774298426,
        "ks": 0.3185893753708038
    },
    "test": {
        "rmse": 0.3768828340618003,
        "mae": 0.2674278041170791,
        "r2": 0.06796332145110762,
        "roc_auc": 0.7057844822870553,
        "pr_auc": 0.36120050919227764,
        "ks": 0.30232074693466676
    },
    "training_time_seconds": 28.17
}


In [8]:
# Save fastai evaluation results
Path("reports").mkdir(exist_ok=True)

with open("reports/results_fastai.json", "w") as f:
    json.dump(fastai_results, f, indent=4)

print("Saved: reports/results_fastai.json")

Saved: reports/results_fastai.json
